In [6]:
import pandas as pd


def expand_date_range(row: pd.Series):
    start = row["Дата.приема.2026"]
    end = row["Дата.увольнения.2026"]

    if pd.isna(start) or pd.isna(end):
        return []

    if end < start:
        return []

    return pd.date_range(start=start, end=end, freq="D").date.tolist()


def transform_initial_plan(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Приведение текстовых колонок
    text_cols = [
        "Подразделение",
        "Источник",
        "ЦФО",
        "Должность",
        "ФИО ",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # Приведение дат
    if "Дата.приема.2026" in df.columns:
        df["Дата.приема.2026"] = pd.to_datetime(
            df["Дата.приема.2026"], errors="coerce"
        ).dt.date

    if "Дата.увольнения.2026" in df.columns:
        df["Дата.увольнения.2026"] = pd.to_datetime(
            df["Дата.увольнения.2026"], errors="coerce"
        ).dt.date

    # Создаем список дат по каждой строке
    df["Дата"] = df.apply(expand_date_range, axis=1)

    # Разворачиваем список дат в строки
    df = df.explode("Дата", ignore_index=True)

    # Удаляем пустые даты после explode
    df = df[df["Дата"].notna()].copy()

    # Приводим Дата к datetime/date
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    # Переупорядочивание колонок
    desired_order = [
        "Подразделение",
        "Источник",
        "ЦФО",
        "Должность",
        "ФИО ",
        "Дата.приема.2026",
        "Дата.увольнения.2026",
        "Дата",
        "Примечание",
    ]
    existing_order = [col for col in desired_order if col in df.columns]
    other_cols = [col for col in df.columns if col not in existing_order]
    df = df[existing_order + other_cols]

    # Переименование
    df = df.rename(
        columns={
            "Дата.приема.2026": "Дата начала работы",
            "Дата.увольнения.2026": "Дата окончания работы",
            "Источник": "План/Факт",
            "ФИО ": "ФИО",
            "Должность": "Профессия",
        }
    )

    # Фильтр только по 2026 году
    df = df[pd.to_datetime(df["Дата"], errors="coerce").dt.year == 2026].copy()

    return df

In [4]:
df_initial_plan_raw = pd.read_excel("./source/Первоначальный план.xlsx", sheet_name="Лист1")

df_initial_plan = transform_initial_plan(df_initial_plan_raw)